# Outliery — wykrywanie i radzenie sobie w pandas / polars / scipy / scikit-learn

**Problem:** "outlier" to nie jeden, precyzyjnie zdefiniowany obiekt — to, co uzna za odstające proste `Z-score`, może zupełnie różnić się od tego, co wykryje metoda wielowymiarowa. Gorzej: najpopularniejsze metody (Z-score, IQR) mają ukryte założenia (symetria rozkładu, brak innych ekstremalnych wartości w danych), które w realnych danych biznesowych często nie są spełnione.

**Porównanie — dwie fundamentalnie różne kategorie:**
- **Jednowymiarowe** (Z-score, zmodyfikowany Z-score, IQR) — patrzą na JEDNĄ kolumnę na raz. Proste, szybkie, ale ślepe na anomalie widoczne dopiero w KOMBINACJI zmiennych.
- **Wielowymiarowe** (odległość Mahalanobisa, `IsolationForest`, `LocalOutlierFactor`, `EllipticEnvelope`, `DBSCAN`) — patrzą na wszystkie kolumny naraz, wykrywają punkty normalne na każdej osi z osobna, ale nietypowe w połączeniu.

**Kiedy które:** jednowymiarowe do szybkiej, interpretowalnej kontroli jakości pojedynczych kolumn (typowe w BI/raportowaniu — "czy ta wartość ma sens"); wielowymiarowe, gdy podejrzewasz, że anomalia ujawnia się dopiero w relacji między zmiennymi (typowe w wykrywaniu oszustw, nietypowych transakcji, błędów systemowych).

## Setup

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
from scipy import stats

rng = np.random.default_rng(21)
n = 200

# Wiek - w miare symetryczny
age = rng.normal(40, 10, n).round(0)
age[5] = 95  # oczywisty outlier

# Dochod - naturalnie prawoskosny (typowe dla danych finansowych)
income = rng.lognormal(mean=8.5, sigma=0.5, size=n).round(0)

df = pd.DataFrame({"age": age, "income": income})
print(f"Skośność age: {df['age'].skew():.2f}, income: {df['income'].skew():.2f}")
df.describe()

## Sekcja 1 — Metoda Z-score: `|z| > 3`

Klasyczna, prosta reguła: standaryzuj kolumnę, oznacz jako outlier wszystko powyżej progu (zwykle `3`). Zakłada rozkład zbliżony do normalnego.

In [ ]:
z_age = np.abs(stats.zscore(df["age"]))
outliers_age = df.index[z_age > 3]
print(f"age: wykryto {len(outliers_age)} outlier(y), indeksy: {outliers_age.tolist()}")

z_income = np.abs(stats.zscore(df["income"]))
outliers_income = df.index[z_income > 3]
print(f"income: wykryto {len(outliers_income)} outlier(y)")

## Sekcja 2 — Metoda IQR (płoty Tukeya)

Granice: `[Q1 - k×IQR, Q3 + k×IQR]`, zwykle `k=1.5`. To ta sama logika, na której opiera się rysowanie "wąsów" na boxplocie.

In [ ]:
def iqr_bounds(s, k=1.5):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr


lo, hi = iqr_bounds(df["age"])
print(f"age granice: [{lo:.1f}, {hi:.1f}], outlierów: {((df['age'] < lo) | (df['age'] > hi)).sum()}")

lo_i, hi_i = iqr_bounds(df["income"])
print(f"income granice: [{lo_i:.1f}, {hi_i:.1f}], outlierów: {((df['income'] < lo_i) | (df['income'] > hi_i)).sum()}")

### Problem: na danych SKOŚNYCH, IQR flaguje naturalny "ogon" rozkładu jako outliery

`income` jest naturalnie prawoskośne (typowe dla dochodów, cen, czasu oczekiwania) — IQR znalazł kilka "outlierów", ale to zwykłe, prawdziwe obserwacje z górnego ogona rozkładu, nie błędy danych.

In [ ]:
flagged = df.loc[df["income"] > hi_i, "income"].sort_values()
print("Wartości oznaczone jako outliery wg IQR (to naturalny ogon rozkładu, nie błędy):")
print(flagged)

## Sekcja 3 — Zmodyfikowany Z-score (mediana + MAD): odporny na efekt maskowania

**Efekt maskowania** — jedna z najgroźniejszych, najmniej znanych pułapek statystyki opisowej: pojedynczy EKSTREMALNY outlier tak bardzo zawyża odchylenie standardowe, że inny, umiarkowany outlier przestaje przekraczać próg `|z| > 3` — mimo że sam w sobie jest realną anomalią.

In [ ]:
age_demo = rng.normal(40, 10, n).round(0)
age_demo[5] = 90    # umiarkowany outlier
age_demo[10] = 500  # BARDZO ekstremalny outlier
s = pd.Series(age_demo)

z = np.abs(stats.zscore(s))
print(f"Zwykły z-score umiarkowanego outliera (90 lat): {z[5]:.2f}  (próg=3, przekracza? {z[5] > 3})")
print(f"Zwykły z-score ekstremalnego outliera (500 lat): {z[10]:.2f}")
print("-> ekstremalna wartość 'zamaskowała' umiarkowaną - ta druga wygląda już na normalną!")

In [ ]:
median = s.median()
mad = stats.median_abs_deviation(s)
modified_z = 0.6745 * (s - median) / mad  # 0.6745 to stała normalizująca dla rozkładu normalnego

print(f"Zmodyfikowany z-score umiarkowanego outliera: {modified_z[5]:.2f}  (próg=3.5, przekracza? {abs(modified_z[5]) > 3.5})")
print(f"Zmodyfikowany z-score ekstremalnego outliera: {modified_z[10]:.2f}")
print("-> mediana i MAD są znacznie mniej wrażliwe na pojedynczą ekstremalną wartość niż średnia/odch. std,")
print("   więc oba outliery zostają poprawnie wykryte")

## Sekcja 4 — Outlier WIELOWYMIAROWY: normalny na każdej osi z osobna

Najważniejsza koncepcja w tej notatce. Zbudujmy dane, gdzie `income` silnie zależy od `age` (starsi zarabiają więcej), a potem dołóżmy punkt: wiek i dochód każdy z osobna w normalnym zakresie, ale ich KOMBINACJA nietypowa (młoda osoba z dochodem typowym dla znacznie starszej).

In [ ]:
age2 = rng.normal(40, 10, n)
income2 = 500 * age2 + rng.normal(0, 2000, n) + 3000
df2 = pd.DataFrame({"age": age2.round(0), "income": income2.round(0)})

# Punkt podstępny: 22 lata (normalny wiek), 32000 dochodu (normalny dochód SAM W SOBIE)
df2.loc[199] = [22, 32000]

z_age2 = np.abs(stats.zscore(df2["age"]))[199]
z_income2 = np.abs(stats.zscore(df2["income"]))[199]
print(f"z-score wieku (22 lata): {z_age2:.2f}  (< 3, wygląda normalnie)")
print(f"z-score dochodu (32000): {z_income2:.2f}  (< 3, wygląda normalnie)")
print("-> ŻADNA z pojedynczych kolumn nie wygląda na outlier - a punkt jest wyraźnie nietypowy!")

### Odległość Mahalanobisa — uwzględnia korelacje między zmiennymi

In [ ]:
from scipy.spatial.distance import mahalanobis

cov = np.cov(df2[["age", "income"]].values.T)
inv_cov = np.linalg.inv(cov)
mean_vec = df2[["age", "income"]].mean().values

distances = df2[["age", "income"]].apply(
    lambda row: mahalanobis(row.values, mean_vec, inv_cov), axis=1
)

print(f"Odległość Mahalanobisa dla podejrzanego punktu: {distances[199]:.2f}")
print(f"Mediana odległości dla reszty danych: {distances[:-1].median():.2f}")
print(f"Ranking tego punktu (1 = najbardziej odstający z {n}): {distances.rank(ascending=False)[199]:.0f}")

## Sekcja 5 — Metody ML: `IsolationForest`, `LocalOutlierFactor`, `EllipticEnvelope`, `DBSCAN`

Cztery różne mechanizmy wykrywania anomalii wielowymiarowych — warto sprawdzić kilka naraz, bo **żadna nie jest uniwersalnie najlepsza** (patrz wynik `LocalOutlierFactor` niżej).

In [ ]:
from sklearn.ensemble import IsolationForest

X = df2[["age", "income"]].values

# IsolationForest: izoluje punkty przez losowe podziały - anomalie izolują się SZYBCIEJ (mniej podziałów)
iso = IsolationForest(contamination=0.02, random_state=0)
pred_iso = iso.fit_predict(X)  # -1 = outlier, 1 = normalny
print(f"IsolationForest - wykryte indeksy: {df2.index[pred_iso == -1].tolist()}")
print(f"Czy podejrzany punkt złapany? {pred_iso[199] == -1}")

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

# LocalOutlierFactor: patrzy na GĘSTOŚĆ LOKALNĄ - wykrywa punkty rzadsze niż ich sąsiedzi
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.02)
pred_lof = lof.fit_predict(X)
print(f"LocalOutlierFactor - wykryte indeksy: {df2.index[pred_lof == -1].tolist()}")
print(f"Czy podejrzany punkt złapany? {pred_lof[199] == -1}")
print("-> NIE złapany w tej konfiguracji! LOF ocenia gęstość WZGLĘDEM SĄSIADÓW,")
print("   więc punkt leżący blisko brzegu 'chmury' normalnych danych może nie wyróżniać się lokalnie,")
print("   mimo że globalnie łamie ogólną zależność między zmiennymi.")

In [ ]:
from sklearn.covariance import EllipticEnvelope

# EllipticEnvelope: zaklada, ze dane pochodza z rozkladu Gaussa (wielowymiarowego) i dopasowuje 'elipse'
ee = EllipticEnvelope(contamination=0.02, random_state=0)
pred_ee = ee.fit_predict(X)
print(f"EllipticEnvelope - wykryte indeksy: {df2.index[pred_ee == -1].tolist()}")
print(f"Czy podejrzany punkt złapany? {pred_ee[199] == -1}")

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

# DBSCAN: klastrowanie gestosciowe - punkty NIE nalezace do zadnego klastra (-1) to 'szum'/outliery
# WYMAGA skalowania (bazuje na odleglosci euklidesowej) - patrz notatka o skalowaniu cech
X_scaled = StandardScaler().fit_transform(X)
db = DBSCAN(eps=0.3, min_samples=5)
labels = db.fit_predict(X_scaled)

print(f"Liczba znalezionych klastrów: {len(set(labels)) - (1 if -1 in labels else 0)}")
print(f"Punktów oznaczonych jako szum: {(labels == -1).sum()}")
print(f"Czy podejrzany punkt jest szumem? {labels[199] == -1}")

## Sekcja 6 — Strategie radzenia sobie z wykrytymi outlierami

### Strategia A — Przycinanie (capping / winsoryzacja)

Zamiast usuwać wiersz, ogranicz wartość do granicy — zachowuje liczbę obserwacji, redukuje wpływ ekstremum. Dobre podejście domyślne, gdy nie masz pewności, czy outlier to błąd, czy rzadka, ale prawdziwa wartość.

In [ ]:
df3 = df.copy()
lo3, hi3 = iqr_bounds(df3["income"])

# pandas
df3["income_capped"] = df3["income"].clip(lower=lo3, upper=hi3)
print(f"pandas - oryginalna max: {df3['income'].max():.0f}, po capping: {df3['income_capped'].max():.0f}")

In [ ]:
# polars - ta sama logika, .clip() na wyrazeniu
df3_pl = pl.from_pandas(df3[["income"]])
result_pl = df3_pl.with_columns(pl.col("income").clip(lo3, hi3).alias("income_capped"))
print(f"polars - max po capping: {result_pl['income_capped'].max():.0f}")

In [ ]:
from scipy.stats.mstats import winsorize

# scipy - przyciecie wg PERCENTYLI (nie IQR) - inna definicja granicy, ten sam mechanizm
winsorized = winsorize(df3["income"], limits=[0.01, 0.01])  # po 1% z każdej strony
print(f"scipy winsorize(1%) - max: {winsorized.max():.0f}")

### Strategia B — Flagowanie (zachowanie oryginalnej wartości + kolumna informacyjna)

Zamiast zmieniać dane, dodaj kolumnę `is_outlier` — zachowuje 100% informacji, pozwala modelowi/analizie świadomie uwzględnić albo odfiltrować te wiersze później. Często najlepsza opcja w kontekście BI/raportowania, gdzie "cichej" zmiany liczb nie chcesz.

In [ ]:
df3["is_outlier"] = (df3["income"] < lo3) | (df3["income"] > hi3)
df3[df3["is_outlier"]]

### Strategia C — Usunięcie

Najbardziej radykalna, nieodwracalna opcja — uzasadniona głównie wtedy, gdy masz PEWNOŚĆ, że to błąd danych (np. wiek `-5` albo `999`), nie rzadka, ale prawdziwa obserwacja. Usuwanie "na wszelki wypadek" zniekształca rozkład i zaniża realną zmienność danych.

In [ ]:
df_clean = df3[~df3["is_outlier"]].drop(columns=["is_outlier"])
print(f"Wierszy przed: {len(df3)}, po usunięciu: {len(df_clean)}")

## Sekcja 7 — Pułapki

### Pułapka 1 — Efekt maskowania (opisany w Sekcji 3)

Zwykły Z-score jest SAM podatny na wpływ outlierów, które ma wykrywać — im bardziej ekstremalna wartość, tym mocniej zawyża odchylenie standardowe używane do wykrycia WSZYSTKICH innych outlierów. Zmodyfikowany Z-score (mediana/MAD) jest znacznie bardziej odporny, bo obie te statystyki są mało wrażliwe na pojedyncze ekstremalne wartości.

### Pułapka 2 — Parametr `contamination` jest arbitralnie zgadywany, a wynik jest od niego prawie liniowo zależny

`IsolationForest`/`LocalOutlierFactor`/`EllipticEnvelope` wymagają z góry podanego `contamination` (oczekiwany % outlierów) — **nie wykrywają "prawdziwej" liczby outlierów same z siebie**. Zmiana tego parametru wprost skaluje liczbę wykrytych punktów.

In [ ]:
for c in [0.01, 0.05, 0.1, 0.2]:
    pred = IsolationForest(contamination=c, random_state=0).fit_predict(X)
    print(f"contamination={c}: wykryto {sum(pred == -1)} outlierów (z {n})")

### Pułapka 3 — IQR/Z-score na danych skośnych flagują naturalny ogon rozkładu (Sekcja 2)

Obie metody zakładają w miarę symetryczny rozkład. Na danych z natury skośnych (dochody, ceny, czasy oczekiwania) będą systematycznie oznaczać prawdziwe, poprawne obserwacje jako "outliery" tylko dlatego, że leżą w długim ogonie. Warto rozważyć transformację (log, Box-Cox — notatka o skalowaniu cech) PRZED wykrywaniem outlierów na takich danych, albo świadomie przyjąć wyższy próg `k` w metodzie IQR.

### Pułapka 4 — Granice przycinania policzone na całym zbiorze (train+test) to ten sam wyciek danych, co przy skalowaniu

Jeśli capping/winsoryzacja jest krokiem przygotowania danych pod model, granice (`Q1`, `Q3`, percentyle) powinny być liczone WYŁĄCZNIE na zbiorze treningowym i zastosowane do zbioru testowego — dokładnie ta sama zasada, co przy `fit`/`transform` skalerów (patrz notatka o skalowaniu cech, Pułapka 1).

## Podsumowanie

| Metoda | Typ | Kiedy stosować | Ograniczenie |
|---|---|---|---|
| Z-score (`\|z\|>3`) | jednowymiarowa | Dane w miarę symetryczne | Podatna na efekt maskowania, zakłada normalność |
| Zmodyfikowany Z-score (mediana/MAD) | jednowymiarowa | Jak wyżej, ale gdy podejrzewasz wiele/silne outliery naraz | Wciąż jednowymiarowa - ślepa na anomalie kombinacji cech |
| IQR (płoty Tukeya) | jednowymiarowa | Szybka kontrola jakości, baza pod boxplot | Flaguje naturalny ogon danych skośnych |
| Mahalanobis | wielowymiarowa | Wykrycie anomalii widocznej dopiero w KOMBINACJI zmiennych, dane w miarę Gaussowskie | Wymaga odwracalnej macierzy kowariancji, wrażliwy na silną współliniowość |
| `IsolationForest` | wielowymiarowa (ML) | Duże zbiory, nieliniowe zależności, brak założeń o rozkładzie | Wymaga zgadnięcia `contamination` |
| `LocalOutlierFactor` | wielowymiarowa (ML) | Anomalie LOKALNE (różna gęstość w różnych regionach danych) | Może przeoczyć anomalie globalne blisko brzegu "chmury" danych |
| `EllipticEnvelope` | wielowymiarowa (ML) | Dane rzeczywiście zbliżone do rozkładu Gaussa | Słaby przy nieliniowych/wielomodalnych rozkładach |
| `DBSCAN` | wielowymiarowa (ML) | Naturalne "skupiska" w danych, outliery jako szum poza klastrami | Wymaga skalowania i strojenia `eps`/`min_samples` |
| Capping/winsoryzacja | strategia obsługi | Domyślnie bezpieczna opcja, zachowuje liczbę wierszy | Zniekształca rozkład ogona |
| Flagowanie | strategia obsługi | Kontekst BI/raportowanie, gdzie liczby nie mogą się "po cichu" zmienić | Wymaga świadomej obsługi flagi w dalszej analizie |
| Usunięcie | strategia obsługi | Tylko przy pewności, że to błąd danych | Zaniża realną zmienność, jeśli nadużywane |

**Wniosek:** żadna metoda wykrywania nie jest uniwersalnie poprawna — `LocalOutlierFactor` w tej notatce przeoczył dokładnie ten punkt, który złapały trzy pozostałe metody wielowymiarowe. Sensowna praktyka to sprawdzenie kilku metod naraz i traktowanie zgodnego wyniku jako mocniejszego sygnału niż pojedyncza metoda — szczególnie przy decyzjach nieodwracalnych (Strategia C: usunięcie).